> **Production version** — functions are imported from `src/mrta/` instead of defined inline.  
> Original teaching version: [`../tutorials/`](../tutorials/)  
> Install first: `pip install -e ".[all]"`


## Production imports (active)

```python
from mrta.prompts import load_prompt, MODES
```

The templates in `src/mrta/prompts/` replace the inline Jinja2 strings defined in this notebook.
See [`../../production-ready.md`](../../production-ready.md) — Phase 08.


# Phase 8 — Teaching Modes & Prompt Engineering

**Goal:** Turn the bare RAG answer into a *teaching* assistant. We design and test five distinct modes, each backed by a Jinja2 prompt template living in `src/mrta/prompts/`:

1. **Beginner** — plain language, analogies, no jargon.
2. **Graduate** — assumes ML coursework background.
3. **Interview** — system-design framing, tradeoff discussion.
4. **Quiz me** — 5 multiple-choice questions with answers.
5. **Lecture notes** — bullet-point study guide with key terms.

All five share the same retrieval + grounding rule. Only the *behavioral* part of the prompt changes. This makes mode switching a one-line config in the UI.


## 8.1 — Why separate prompt files?

Prompts are configuration, not code. Putting them in `src/mrta/prompts/*.jinja2`:

- lets non-engineers iterate without touching Python;
- gives you a clean diff history of every prompt change in `git log`;
- makes A/B testing two prompt versions trivial (`load_prompt("beginner_v2")`).


In [ ]:
import os, sys, json
from pathlib import Path

repo_root = Path.cwd()
while not (repo_root / "pyproject.toml").is_file() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
os.chdir(repo_root)
sys.path.insert(0, str(repo_root))

PROMPTS = Path("src/mrta/prompts"); PROMPTS.mkdir(parents=True, exist_ok=True)
print("Prompts live at:", PROMPTS.resolve())


## 8.2 — The shared base template

Every mode renders `_base.jinja2` and overrides the `{% block behavior %}` and `{% block format %}` slots.


In [ ]:
# Template: see src/mrta/prompts/_base.j2


## 8.3 — Mode templates


In [ ]:
# Templates: see src/mrta/prompts/ (beginner, expert, interview, quiz, lecture_notes)


## 8.4 — Loader


In [ ]:
from mrta.prompts import load_prompt, MODES

# Smoke test:
preview = load_prompt("beginner", question="What is attention?", chunks=[
    {"source": "aiayn.pdf", "page": 4, "text": "Scaled dot-product attention computes ..."}
])
print(preview[:600])


## 8.5 — Run all five modes on the same question

Same retrieval, same context — only the behavioral block changes. Watch how the same answer reshapes itself.


In [ ]:
import faiss
from sentence_transformers import SentenceTransformer
from pydantic import BaseModel
from typing import Optional
import ollama, time, re
from mrta.retrieval import VectorStore, Embedder
from mrta.core.config import settings

# class VectorStore:
#     def __init__(self, dim, embedder):
#         self.index = faiss.IndexFlatIP(dim); self.metadata=[]; self.embedder=embedder
#     def search(self, q, k=5):
#         import numpy as np
#         v = self.embedder.encode([q], normalize_embeddings=True).astype("float32")
#         s, i = self.index.search(v, k)
#         return [{**self.metadata[idx], "score": float(s[0][r])} for r, idx in enumerate(i[0]) if idx >= 0]
#     @classmethod
#     def load(cls, p, embedder):
#         p = Path(p); cfg = json.loads((p/"config.json").read_text())
#         st = cls(cfg["dim"], embedder)
#         st.index = faiss.read_index(str(p/"index.faiss"))
#         st.metadata = [json.loads(l) for l in (p/"metadata.jsonl").read_text().splitlines()]
#         return st

embedder = Embedder(settings.embedding_model)    # picks up nomic-embed-text from dev.yaml
store = VectorStore.load("data/vector_store/aiayn", embedder)

# embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
# store = VectorStore.load("data/vector_store/aiayn", embedder)

def ask(mode: str, question: str, k: int = 5) -> str:
    hits = store.search(question, k=k)
    prompt = load_prompt(mode, question=question, chunks=hits)
    r = ollama.chat(model="llama3.2:latest",
                    messages=[{"role": "system", "content": "Follow the instructions exactly."},
                              {"role": "user", "content": prompt}],
                    options={"temperature": 0.2 if mode != "quiz" else 0.5})
    return r["message"]["content"]

question = "How does scaled dot-product attention work?"

for mode in ["beginner", "expert", "interview", "quiz", "lecture_notes"]:
    print("=" * 72)
    print(f"MODE: {mode}")
    print("=" * 72)
    print(ask(mode, question)[:1200])
    print()


## 8.6 — Prompt-quality checks

When you change a prompt, run a fixed set of questions and diff the outputs. This is cheap and catches regressions early. A minimal check:

```python
def smoke_test(mode):
    fixtures = [
        "What is multi-head attention?",
        "How is positional information encoded?",
        "What dataset was used and what BLEU was achieved?",
    ]
    return {q: ask(mode, q, k=4) for q in fixtures}
```

Store the outputs under `tests/prompt_fixtures/<mode>.md` and diff them when you tweak a template.


## 8.7 — Heuristics that help on weak models

`llama3.2:latest` is small. A few prompt tricks substantially improve grounding:

1. **State the rule twice.** Once at the top, once just before the answer. Small models forget mid-context.
2. **Number the chunks** in the prompt — small models follow numbered references more reliably than free-form `[page]`.
3. **Pre-emptive refusal.** Add: *"If you cannot find the answer, the correct response is exactly: I could not find this in the provided documents."*
4. **Forbid hedging when grounded.** Add: *"Do not say 'maybe' or 'might'. If you have evidence, state it."*

Move to a larger model (`llama3.1:8b`, `qwen2.5:7b`) only when these heuristics stop being enough.


## What you learned

- How to factor mode behavior into a shared Jinja2 base template.
- Five concrete teaching-mode prompts you can drop into the app.
- A loader that wires modes through a `MODES` dict.
- Prompt-quality smoke testing.
- Small-model prompting heuristics.

## Exercises

1. Add a "summarize this paper for my advisor" mode that outputs a 3-paragraph summary with citations.
2. Add a "code explanation" mode that, given a code snippet in the context, walks through it line by line.
3. Write a prompt-fixture test: assert that the `beginner` mode never includes the substring "softmax" without first defining it.

## Next: [Phase 9 — Evaluation, logging & Docker](./2026-05-25-phase09-evaluation-logging-docker.ipynb)
